In [ ]:
import os
import warnings
import itertools
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime
import time
timestr = time.strftime("%Y%m%d-%H%M%S")
import joblib

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold


# Modeling


In [2]:
from google.colab import drive
drive.mount('/content/drive')
# list files in the folder
print(os.listdir("/content/drive/MyDrive/DMT_data"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['test_set_VU_DM.csv', 'training_set_VU_DM.csv', 'DMT_data']


In [3]:
!ln -s "/content/drive/MyDrive/DMT_data" /content/DMT_data
# This should now show your data
!ls /content/DMT_data

ln: failed to create symbolic link '/content/DMT_data/DMT_data': File exists
DMT_data  test_set_VU_DM.csv  training_set_VU_DM.csv


Plot style

In [4]:
plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.color":        "#e4e4e4",
    "grid.linewidth":    0.05,
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.titleweight":  "bold",
})

BLUE   = "#378ADD"
TEAL   = "#1D9E75"
AMBER  = "#EF9F27"
CORAL  = "#D85A30"
PURPLE = "#7F77DD"
GRAY   = "#888780"

## Read data

In [ ]:
data_path = '/content/DMT_data'
train_file = os.path.join(data_path, 'training_set_VU_DM.csv')
test_file = os.path.join(data_path, 'test_set_VU_DM.csv')

train_df = pd.read_csv(train_file)
df = train_df.copy()
display(df.head())
#test = pd.read_csv(test_file)

,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,893,3,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
1,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,10404,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
2,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,21315,3,4.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
3,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,27348,2,4.0,...,NaN,NaN,NaN,NaN,-1.0,0.0,5.0,0,NaN,0
4,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,29604,4,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0


# ADD: a function with all preprecessing steps we did

In [ ]:
def preprocessing(data):
    #preprocessing for both train and test

    # preprocessing of target varibales (which will not be present in the test file) - code will run only if the column exists
    if 'booking_bool' in data.columns:
        # ADD: potential cleaning functions we want here
        pass


    return df
df = preprocessing(df)

Read to see what are the options:
https://towardsdatascience.com/a-complete-guide-to-recommender-system-tutorial-with-sklearn-surprise-keras-recommender-5e52e8ceace1/

## K-Nearest Neighbours
https://www.geeksforgeeks.org/machine-learning/recommender-systems-using-knn/


Content-Based Recommender System - but we can add use features and make it hybrid

In [ ]:
# 1. feature selection - for now only a few features to see how the model performs
features = [
    'prop_starrating', 'prop_review_score', 'prop_location_score1',
    'prop_location_score2', 'prop_log_historical_price',
    'price_usd', 'promotion_flag'
]

# 2. Preprocessing Pipeline
# extract features and handle missing values/scaling
X = df[features].copy()

# data imputation
imputer = SimpleImputer(strategy='median') #missing values with median
X_processed = imputer.fit_transform(X)

#scaling
scaler = StandardScaler() #scaling
X_scaled = scaler.fit_transform(X_processed)

# 3. train-test split (80-20)
# training set to find centroids; test set to evaluate assignment stability
X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)


# Elbow method to find the best k
inertia = []
K_range = range(1, 30)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

# Plotting the Elbow
plt.figure(figsize=(8, 4))
plt.plot(K_range, inertia, 'bx-')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method For Optimal k')
plt.show()

In [ ]:

# 4. unsupervised Cross-Validation (inertia stability)
# we use K-Fold to ensure the clusters (inertia) are consistent across different slices of data
kf = KFold(n_splits=5, shuffle=True, random_state=42)
k_clusters = 30 #change for the optimal
cv_inertia = []

print(f"Starting 5-Fold Cross-Validation for K={k_clusters}...")

for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train[train_index], X_train[val_index]

    # fit model on the fold training data
    km_fold = KMeans(n_clusters=k_clusters, n_init=5, random_state=42)
    km_fold.fit(X_fold_train)

    # calculate Inertia on the validation fold
    # inertia = sum of squared distances of samples to their closest cluster center
    cv_inertia.append(km_fold.inertia_)

print(f"Average CV Inertia: {np.mean(cv_inertia):.2f} (+/- {np.std(cv_inertia):.2f})")


In [ ]:
# 1. feature selection - for now only a few features to see how the model performs
features = [
    'prop_starrating', 'prop_review_score', 'prop_location_score1',
    'prop_location_score2', 'prop_log_historical_price',
    'price_usd', 'promotion_flag'
]

# 2. Preprocessing Pipeline
# extract features and handle missing values/scaling
X = df[features].copy()  #TO DO: add more features we want to model

# data imputation
imputer = SimpleImputer(strategy='median') #missing values with median
X_processed = imputer.fit_transform(X)

#scaling
scaler = StandardScaler() #scaling
X_scaled = scaler.fit_transform(X_processed)

# 3. train-test split (80-20)
# training set to find centroids; test set to evaluate assignment stability
X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)

# 4. unsupervised Cross-Validation (inertia stability)
# we use K-Fold to ensure the clusters (inertia) are consistent across different slices of data
kf = KFold(n_splits=5, shuffle=True, random_state=42)
k_clusters = 50
cv_inertia = []

print(f"Starting 5-Fold Cross-Validation for K={k_clusters}...")

for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train[train_index], X_train[val_index]

    # fit model on the fold training data
    km_fold = KMeans(n_clusters=k_clusters, n_init=5, random_state=42)
    km_fold.fit(X_fold_train)

    # calculate Inertia on the validation fold
    # inertia = sum of squared distances of samples to their closest cluster center
    cv_inertia.append(km_fold.inertia_)

print(f"Average CV Inertia: {np.mean(cv_inertia):.2f} (+/- {np.std(cv_inertia):.2f})")

Starting 5-Fold Cross-Validation for K=50...
Average CV Inertia: 2647813.22 (+/- 20324.80)


In [7]:
# 5. final model implementation
# fitting on the full training set and assigning clusters to the test set
final_kmeans = KMeans(n_clusters=k_clusters, n_init=10, random_state=42)
final_kmeans.fit(X_train)

# assign cluster labels back to the original dataframe for the test indices
train_indices, test_indices = train_test_split(df.index, test_size=0.2, random_state=42)
df.loc[train_indices, 'hotel_cluster'] = final_kmeans.labels_
df.loc[test_indices, 'hotel_cluster'] = final_kmeans.predict(X_test)

print("\nCluster assignment complete. Sample distribution:")
print(df['hotel_cluster'].value_counts().head())


Cluster assignment complete. Sample distribution:
hotel_cluster
17.0    309776
2.0     306708
40.0    294968
43.0    199867
23.0    198242
Name: count, dtype: int64


#### Computing the predicted ranking from the clusters - pointwise ranking
Goal: "rank the properties belonging to a user search on the likeliness that the property will be booked" for each user list all the properites from their search based on the probability that that property will be booked, i.e. highest probability that booking_bool=1

The evaluation function is (NDCG)@5 calculated per query and averaged over all queries with the values weighted by the log_2 function - this rewards ehavily if the acutal booked hotel appears at the very top of our list.

Since K-means is unsupervised, the "Cluster ID" itself doesn't tell "probability." We have to map the clusters to historical booking success. The way to do this:

1. Calculate the *Mean Booking Rate* for each cluster.

2. Assign this "Cluster Score" to every hotel in that cluster.

3. Sort the hotels within each SearchId by that score.


cluster_score - Probability Estimate derived from historical data. total number of bookings in a cluster/total number of hotel listings which were put in that cluster e.g if Cluster 5 has 1,000 hotel listings and 50 of them were booked, its cluster_score is 0.05.

In [10]:
# visualising the differences between the culsters - average attributes for each cluster
cluster_profiles = df.groupby('hotel_cluster')[features + ['booking_bool']].mean()

print("Cluster profiles by booking probability:")
display(cluster_profiles.sort_values(by='booking_bool', ascending=False).head(10)) # sort by booking_bool - most booked clusters

Cluster Profiles (Sorted by Booking Probability):


,prop_starrating,prop_review_score,prop_location_score1,prop_location_score2,prop_log_historical_price,price_usd,promotion_flag,booking_bool
hotel_cluster,,,,,,,,
3.0,2.853198,3.902936,2.253529,0.580634,4.769558,1.290011e+02,0.009174,0.121240
5.0,4.636364,3.954545,3.462727,0.056500,2.006364,4.285486e+06,0.272727,0.090909
20.0,2.868457,4.050294,2.476815,0.259172,4.830187,1.276556e+02,0.000000,0.069983
49.0,3.659463,4.023662,4.782237,0.649141,5.361562,1.921578e+02,1.000000,0.063155
33.0,3.185057,3.783923,3.794737,0.265938,5.007226,1.299185e+02,1.000000,0.061267
36.0,3.639277,4.010657,4.618811,0.448703,0.000000,2.109575e+02,1.000000,0.055184
34.0,4.184211,4.263158,3.245526,0.071414,3.907105,2.552001e+06,0.368421,0.052632
26.0,3.513439,4.074922,4.210516,0.467427,0.000000,2.457806e+02,0.000000,0.049329
18.0,3.962934,4.156162,4.793386,0.375484,5.458065,2.247765e+02,1.000000,0.044785


In [ ]:
# 1. calculate the cluster_score of each cluster based on historical bookings
cluster_weights = df.groupby('hotel_cluster')['booking_bool'].mean().reset_index()
cluster_weights.columns = ['hotel_cluster', 'cluster_score']

# 2. merge this score back to into the df
final_df = df.merge(cluster_weights, on='hotel_cluster', how='left')

# 3. double sorting to handle tie-breakers
# if two hotels are in the same cluster, they have the same cluster_score.
# we use prop_review_score as a secondary sort to break ties - we can consider sorting first by a differnt feature to break ties (for exmaple the location)
final_df = final_df.sort_values(by=['srch_id', 'cluster_score', 'prop_review_score'], ascending=[True, False, False])


# save file
final_df.to_csv(f'VU-DM-2026-Group-62-{timestr}.csv', index=False)

print("K-means Ranker Submission Generated.")
print(final_df.head(30).to_string(index=False))

## Saving the model, scaler and imputer 
in case sth crashes - then we can recover the model and use for submission

In [ ]:
def save_model_bundle(model, scaler, imputer, weights, folder='models'):
    # create directory if it doesn't exist
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    date_str = datetime.now().strftime("%Y%m%d")  #timestamped filename
    
    # save each component - joblib is more efficient for large files
    joblib.dump(model, f'{folder}/kmeans_{date_str}.pkl')
    joblib.dump(scaler, f'{folder}/scaler_{date_str}.pkl')
    joblib.dump(imputer, f'{folder}/imputer_{date_str}.pkl')
    joblib.dump(weights, f'{folder}/weights_{date_str}.pkl')
    
    print(f"Model, scaler, imputer and cluster weigths saved in /{folder} with date {date_str}")

save_model_bundle(final_kmeans, scaler, imputer, cluster_weights)

# Test file - final prediction


In [ ]:
#1. load and prepare the data
data_path = '/content/DMT_data'
test_file_df = pd.read_csv(os.path.join(data_path, 'test_set_VU_DM.csv'))

#apply the pre-processing function
#TO DO
test_file_df = preprocessing(test_file_df)

# select same features used in training
X_test_file = test_file_df[features].copy()

# apply the imputer and scaler - the already fitted one (do not re-fit)
X_test_file_imputed = imputer.transform(X_test_file)
X_test_file_scaled = scaler.transform(X_test_file_imputed)

# 2. assign clusters - we use .predict() to see which training cluster these new rows belong to
test_file_df['hotel_cluster'] = final_kmeans.predict(X_test_file_scaled) #final_kmeans is the model we fitted before

# 3. apply the cluster scores we calculated from the training
test_file_df = test_file_df.merge(cluster_weights, on='hotel_cluster', how='left') #cluster_weigths are the probabilities of each cluster

# in case any clsuters in tests were not present in training we fill them with global mean
test_file_df['cluster_score'] = test_file_df['cluster_score'].fillna(df['booking_bool'].mean())

# 4. ouble sorting to handle tie-breakers
test_file_ranked = test_file_df.sort_values(by=['srch_id', 'cluster_score', 'prop_review_score'], ascending=[True, False, False])

test_file_ranked.to_csv(f'VU-DM-2026-Group-62-{timestr}.csv', index=False) #save the submission file

# XGBoost
or a different model

##